# Spark Exercise

Apache Spark is an excellent tool for data engineering projects due to its robust ability to process large-scale data efficiently through distributed computing. Spark's in-memory processing capabilities significantly enhance the speed of data operations, making it ideal for handling big data workloads. It supports various data sources and formats, offering versatility in data ingestion and transformation. Additionally, Spark's rich API supports multiple programming languages such as Python, Java, and Scala, catering to diverse developer preferences. Its ecosystem, which includes libraries for SQL, machine learning, and graph processing, provides a comprehensive suite for building complex data pipelines and analytics, making it a powerful and flexible choice for data engineering tasks.

Use Python, ```pyspark``` and ```pandas``` to explore Apache Spark RDD and DataFrame:

# Spark RDD

Spark RDD (Resilient Distributed Dataset) is a fundamental data structure in Apache Spark that enables fault-tolerant, distributed processing of large datasets across multiple nodes in a cluster. Spark RDDs provide a higher-level abstraction for performing distributed data processing tasks, including both map (transformations) and reduce (aggregations) operations.

## Import Necessary Libraries

In [1]:
import csv
import os
import shutil
import sys
from io import StringIO

import pandas as pd
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, count

## Spark Context and Session
Initialize Spark Context and Spark Session

In [2]:
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)

csv_file = "cleaned-amazon-products-electronics-sales-2023.csv"

conf = SparkConf()
conf = conf.setMaster("local[2]")
conf = conf.setAppName("bdeng-spark")
conf = conf.set("spark.ui.showConsoleProgress", "false")

sc = SparkContext.getOrCreate(conf=conf)
sc.setLogLevel("WARN")

spark_builder = SparkSession.builder
spark_builder = spark_builder.appName("spark")
spark = spark_builder.getOrCreate()

print(f"Spark {sc.version} running on {sc.master}")

def parse_csv_line(line):
    line_as_file = StringIO(line)
    csv_reader = csv.reader(line_as_file)
    row = next(csv_reader)
    return row

def get_rating_pair(row):
    rating = row[5]
    return rating, 1

def add_numbers(first_number, second_number):
    return first_number + second_number

def sort_by_rating(item):
    rating = item[0]
    return rating

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/08 00:30:10 WARN Utils: Your hostname, LaggerThinkPad, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/08 00:30:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/08 00:30:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.1.2 running on local[2]


## Load Data into RDD

In [3]:
lines_rdd = sc.textFile(csv_file)
header = lines_rdd.first()

def is_not_header(line):
    return line != header

data_lines_rdd = lines_rdd.filter(is_not_header)
products_rdd = data_lines_rdd.map(parse_csv_line)

print(products_rdd.count())

9411


## Map Operation

Split data into individual parts and create key-value pairs

In [4]:
rating_pairs_rdd = products_rdd.map(get_rating_pair)

print(rating_pairs_rdd.take(5))

[('4.0', 1), ('4.3', 1), ('4.2', 1), ('4.1', 1), ('4.3', 1)]


## Reduce Operation

Reduce your key-value pairs

In [5]:
rating_counts_rdd = rating_pairs_rdd.reduceByKey(add_numbers)

## Collect Results

Because of lazy evaluation, the map-reduce operation is performed only now. Show what you calculated.

In [6]:
rating_counts = rating_counts_rdd.collect()
rating_counts = sorted(rating_counts, key=sort_by_rating)

# Number of products per rating
for rating, product_count in rating_counts:
    print(rating, product_count)

1.0 8
1.3 1
1.4 1
1.5 2
1.7 2
1.8 1
1.9 1
2.0 2
2.1 3
2.2 4
2.3 3
2.4 6
2.5 7
2.6 9
2.7 7
2.8 18
2.9 25
3.0 44
3.1 45
3.2 61
3.3 102
3.4 162
3.5 203
3.6 309
3.7 397
3.8 610
3.9 793
4.0 973
4.1 1155
4.2 1209
4.3 1301
4.4 873
4.5 583
4.6 251
4.7 95
4.8 39
4.9 9
5.0 97


## Save Results

In [7]:
rating_counts_rdd.saveAsTextFile("saved-rating-rdd")
print("RDD saved.")

Py4JJavaError: An error occurred while calling o120.saveAsTextFile.
: org.apache.hadoop.mapred.FileAlreadyExistsException: Output directory file:/mnt/c/Users/lagge/Documents/projects-FHTW-SS26/project-BDENG/BigDataEngineeringProject/exercise-spark/saved-rating-rdd already exists
	at org.apache.hadoop.mapred.FileOutputFormat.checkOutputSpecs(FileOutputFormat.java:131)
	at org.apache.spark.internal.io.HadoopMapRedWriteConfigUtil.assertConf(SparkHadoopWriter.scala:303)
	at org.apache.spark.internal.io.SparkHadoopWriter$.write(SparkHadoopWriter.scala:73)
	at org.apache.spark.rdd.PairRDDFunctions.$anonfun$saveAsHadoopDataset$1(PairRDDFunctions.scala:1094)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.scala:18)
	at org.apache.spark.rdd.RDDOperationScope$.withScope(RDDOperationScope.scala:151)
	at org.apache.spark.rdd.RDDOperationScope$.withScope(RDDOperationScope.scala:112)
	at org.apache.spark.rdd.RDD.withScope(RDD.scala:417)
	at org.apache.spark.rdd.PairRDDFunctions.sav

# Spark DataFrame

Spark DataFrame is a distributed collection of data organized into named columns, designed for efficient data manipulation and analysis in Apache Spark. It is used for various data processing tasks such as data ingestion, transformation, querying, and analysis in Apache Spark, providing a high-level abstraction that simplifies working with structured data.

## Load Data into DataFrame

In [ ]:
pd.options.display.max_rows = 5

data = pd.read_csv(
    csv_file,
    sep=",",
    header=0,
    dtype={
        "name": str,
        "main_category": str,
        "sub_category": str,
        "image": str,
        "link": str,
        "ratings": float,
        "no_of_ratings": int,
        "discount_price": float,
        "actual_price": float,
    },
)

df = spark.createDataFrame(data)

## View DataFrame Schema

In [ ]:
df.printSchema()

## View DataFrame Data

In [ ]:
df.show(10)

## Filter Data

Performe a filter operation on a column

In [ ]:
cheap_products_df = df.filter(df.discount_price < 20.0)

print("Count of all products with discount price < 20 EUR:", cheap_products_df.count())
cheap_products_df.show(10)

## Group By and Aggregate

Performe a group by and aggregat operation

In [ ]:
grouped_by_rating_df = df.groupBy(df.ratings)

rating_summary_df = grouped_by_rating_df.agg(
    count("*").alias("product_count"),
    avg(df.discount_price).alias("avg_discount_price"),
)

rating_summary_df = rating_summary_df.sort("ratings")

rating_summary_df.show()

## Save DataFrame to Parquet

In [ ]:
df.write.parquet("saved-products-dataframe")
print("DataFrame saved.")

In [ ]:
spark.stop()